# 03 — Semantic Narrative Search

Implementing cosine similarity search using the pre-computed contextual embeddings. The goal is to retrieve messages that are semantically closest to the manually defined narratives.

In [5]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util

# Loading dataset and embeddings
df = pd.read_csv('aletheia_filtered.csv')
embeddings = np.load('embeddings_filtered.npy')

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
device = model.device
print(f'Running on: {device}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Running on: mps:0


### 1. Narrative Definition and Encoding

In [6]:
queries = [
    "Vacinas causam doenças autoimunes",
    "Vacinas contêm microchips ou substâncias ocultas",
    "Governos escondem efeitos colaterais das vacinas",
    "Imunidade natural é superior à vacinal",
    "Vacinas foram desenvolvidas rápido demais e não são seguras"
]

q_embeddings = model.encode(queries, convert_to_tensor=True)
corpus_t = torch.from_numpy(embeddings).to(device)

### 2. Similarity Calculation and Ranking

In [7]:
search_results = {}

for i, text in enumerate(queries):
    # Calculate similarity between current query and the whole corpus
    scores = util.cos_sim(q_embeddings[i], corpus_t)[0]
    
    # Get top 10 matches
    top_k = torch.topk(scores, k=10)
    
    # Storing results as a dictionary
    search_results[text] = {
        'indices': top_k.indices.tolist(),
        'scores': top_k.values.tolist()
    }
    print(f'Search completed for: {text}')

Search completed for: Vacinas causam doenças autoimunes
Search completed for: Vacinas contêm microchips ou substâncias ocultas
Search completed for: Governos escondem efeitos colaterais das vacinas
Search completed for: Imunidade natural é superior à vacinal
Search completed for: Vacinas foram desenvolvidas rápido demais e não são seguras


### 3. Result Inspection

In [8]:
for q, res in search_results.items():
    print(f'\n>>> NARRATIVE: {q}')
    print('-' * 40)
    
    # Unpacking lists from the dictionary
    for idx, s in zip(res['indices'], res['scores']):
        # Ensuring content is string to avoid NaN errors
        content = str(df.iloc[idx]['text_content'])
        preview = (content[:150] + '...') if len(content) > 150 else content
        print(f'[{s:.4f}] {preview}\n')


>>> NARRATIVE: Vacinas causam doenças autoimunes
----------------------------------------
[0.8406] Autoimunidade induzida por vacina: o papel do mimetismo molecular e da reação imunológica cruzada.
Isso explica o aumento dos casos de doenças autoimu...

[0.7245] PADRÃO GENOCIDA

💉🧬🐍🧲🔊🧟‍♂️☠️⚰️

Dra. Maria Emília Gadelha afirma que a proteína spike das vacinas tem um modus operandis, um padrão que provoca lesões...

[0.7216] Eu não sou anti vacina não mas esse vídeo está dizendo que vacinas como a de sarampo evitam o contágio da doença, isso não é verdade. Eu tomei todas a...

[0.7115] Novo problema surge para os vacinados contra COVID

Alguns estudos sugerem que uma reação autoimune induzida pela proteína spike pode ser a culpada.

...

[0.7070] "Em Israel 4 em cada 5 internados, ESTÃO VACINADOS"

RTP3 - canal português.

Será que as vacinas, estão a provocar a infeção de pessoas por meio de A...

[0.6956] Nem todo ser vai associa a vacinaçã com esses mal súbitos infelizmente , iram co